# Pretraining and the Model Landscape — companion notebook: subword tokenization

Chapter 8 of *Applied Natural Language Processing — From Tokens to Agents*.
Read the chapter first: **[Pretraining and the Model Landscape](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/08-pretraining-landscape.html)**.

This notebook covers one section of the chapter, **[Subword tokenization: how a model sees text](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/08-pretraining-landscape.html#sec-subword)**.
The section used to be part of chapter 2 and moved here, next to the pretrained models that
use these tokenizers; the cells moved with it. The rest of chapter 8 (the data funnel, the
compute arithmetic, the model landscape) has no notebook cells yet.

The chapter's status note says every number and code output in this section came from
running code. This notebook runs that code, in the section's order:

1. **Byte-pair encoding from scratch**: learning merges on a four-word corpus, and replaying
   them on a word that was never seen.
2. **Characters or bytes**: what BPE starts from, UTF-8, byte-level tokens that cut through
   a character, and byte fallback.
3. **Who pays for Turkish**: one text in two languages under OpenAI's tokenizers, a Turkish
   word as its suffixes stack up, and the four stages of a real tokenizer.
4. **Characters, subwords, whole words** on one small corpus, and the deep dive on WordPiece
   and Unigram LM.
5. **When WordPiece says `[UNK]`**, from the chapter's interview answer.

**How to read the numbers.** Everything printed below is what *this run* produced. Where
the chapter prints the same output, the cell prints the book's value beside it, or compares
the whole printed block line by line, and says whether the two match. Where the chapter
prints an output but not the code that made it, the code here is our reading of the prose,
and the cell says so.

**Interactive figures.** The chapter's BPE merge replay is an Observable widget and cannot
run in a notebook; the cells in section 1 print the same merges.

**Runtime.** A few seconds on a laptop CPU after the first download: two `tiktoken`
vocabularies (about 5 MB together) and three tokenizer files from the Hugging Face Hub
(about 13 MB together). No cell is heavy, so `ANLP_SKIP_HEAVY=1` changes nothing here.

## 0. Setup

The first cell records the interpreter and package versions. The chapter's status note
names `tiktoken` 0.13, `sentencepiece` 0.2.2 and `tokenizers` 0.22; the
book's own measurements ran on Python
3.14.5 ([Appendix A](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/appendices/a-setup.html)). If your numbers differ from the ones saved here, compare this
cell first.

Downloads are cached under `data/cache/` at the root of the notebooks repository, which git
ignores, so a second run reads them from disk. Run outside the repository, the notebook
caches them under `data/cache/` next to itself instead.

In [1]:
import io
import math
import os
import platform
import re
import sys
import time
import unicodedata
from collections import Counter
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

here = Path.cwd()
repo_root = next((p for p in [here, *here.parents] if (p / "scripts" / "check-structure.py").exists()), here)
CACHE = repo_root / "data" / "cache"
os.environ.setdefault("TIKTOKEN_CACHE_DIR", str(CACHE / "tiktoken"))  # tokenizer vocabularies
DATA_HOME = str(CACHE / "scikit_learn_data")                           # the 20 Newsgroups archive
os.environ.setdefault("HF_HOME", str(CACHE / "huggingface"))          # tokenizer files from the Hub
os.environ.setdefault("HF_HUB_VERBOSITY", "error")

print(f"Python        {platform.python_version()}  ({platform.system()} {platform.machine()})")
for pkg in ["numpy", "tiktoken", "sentencepiece", "tokenizers", "matplotlib"]:
    print(f"{pkg:<13} {version(pkg)}")

Python        3.12.13  (Darwin arm64)
numpy         2.5.2
tiktoken      0.13.0
sentencepiece 0.2.2
tokenizers    0.22.2
matplotlib    3.11.2


Two helpers do the comparing. `vs_book` puts one number beside the book's, formatted at the
book's printed precision. `record()` starts keeping a copy of everything a cell prints, and
`compare_with_book(...)` checks that copy against the output block the chapter prints. It
says *line for line* when every character matches, and *only the spacing differs* when the
values match but the columns are padded differently (the book sometimes prints NumPy
arrays with the padding removed).

In [2]:
def vs_book(label, ours, book, fmt="{:,}"):
    """Print this run's value beside the book's, compared at the book's printed precision."""
    ours_s, book_s = fmt.format(ours), fmt.format(book)
    verdict = "same as the book" if ours_s == book_s else "DIFFERS from the book"
    print(f"{label:<38} this run {ours_s:>9}   book {book_s:>9}   {verdict}")


class _Tee(io.TextIOBase):
    """Writes to the notebook as usual and keeps a copy."""

    def __init__(self, stream):
        self.stream, self.copy = stream, io.StringIO()

    def write(self, s):
        self.copy.write(s)
        self.stream.write(s)
        return len(s)

    def flush(self):
        self.stream.flush()


def record():
    """Start keeping a copy of everything printed, to compare with the book's output."""
    if isinstance(sys.stdout, _Tee):
        sys.stdout = sys.stdout.stream
    sys.stdout = _Tee(sys.stdout)


def _squash(lines):
    """Collapse runs of spaces, and the padding NumPy puts inside brackets."""
    return [re.sub(r"\[ +| +\]", lambda m: m.group().strip(), " ".join(line.split())) for line in lines]


def compare_with_book(book_text):
    """Stop recording, and compare what was printed with the output block the book prints."""
    if not isinstance(sys.stdout, _Tee):
        raise RuntimeError("call record() before the code whose output you want to compare")
    ours_text = sys.stdout.copy.getvalue()
    sys.stdout = sys.stdout.stream
    ours = [line.rstrip() for line in ours_text.strip("\n").splitlines()]
    book = [line.rstrip() for line in book_text.strip("\n").splitlines()]
    print()
    if ours == book:
        print(f"-> same as the book, line for line ({len(book)} lines)")
    elif _squash(ours) == _squash(book):
        print(f"-> same values as the book ({len(book)} lines); only the spacing differs")
    else:
        print("-> DIFFERS from the book:")
        for k in range(max(len(ours), len(book))):
            o = ours[k] if k < len(ours) else "(no line)"
            b = book[k] if k < len(book) else "(no line)"
            if " ".join(o.split()) != " ".join(b.split()):
                print(f"   line {k + 1}\n     this run: {o}\n     book    : {b}")

## 1. Byte-pair encoding from scratch

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/08-pretraining-landscape.html#sec-text-as-data-bpe). Before the algorithm, the chapter shows one
unseen word at three unit sizes. As a whole word it is `<UNK>`. As characters nothing is
lost, but the word becomes long and no unit says anything. As subwords it is three pieces
the model has met in other words. The English edition uses `unbreakable` for this, the
Turkish edition `evlerden` ("from the houses"). The chapter says the subword row is how
`cl100k_base` really splits the word, and the cell checks that for both words.

The first use of `tiktoken` downloads the encoding's vocabulary file; section 3 says where
it is cached.

In [3]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
for word, book in [("unbreakable", "un break able"), ("evlerden", "ev ler den")]:
    print(f"{word}: as characters, {len(word)} units: {' '.join(word)}")
    vs_book("  as subwords (cl100k_base)", " ".join(enc.decode([t]) for t in enc.encode(word)), book, "{}")

unbreakable: as characters, 11 units: u n b r e a k a b l e
  as subwords (cl100k_base)            this run un break able   book un break able   same as the book
evlerden: as characters, 8 units: e v l e r d e n
  as subwords (cl100k_base)            this run ev ler den   book ev ler den   same as the book


Now the algorithm that finds such pieces. Start with characters, count every adjacent pair,
merge the most frequent pair into a new symbol, repeat. The code is the chapter's, line for
line.

**What to look at:** the first three merges build `est</w>`, the English superlative with
its word boundary, and nobody told the algorithm that English has suffixes.

In [4]:
record()
from collections import Counter

corpus = {"low": 5, "lower": 2, "newest": 6, "widest": 3}
words = {tuple(w) + ("</w>",): n for w, n in corpus.items()}

def pair_counts(words):
    pairs = Counter()
    for symbols, freq in words.items():
        for a, b in zip(symbols, symbols[1:]):
            pairs[(a, b)] += freq
    return pairs

def merge(words, pair):
    a, b = pair
    out = {}
    for symbols, freq in words.items():
        new, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new.append(a + b)
                i += 2
            else:
                new.append(symbols[i])
                i += 1
        out[tuple(new)] = freq
    return out

merges = []
for step in range(1, 11):
    best, count = pair_counts(words).most_common(1)[0]
    merges.append(best)
    words = merge(words, best)
    print(f"merge {step:>2}: {best[0]!r} + {best[1]!r} -> "
          f"{best[0] + best[1]!r}  (seen {count} times)")
compare_with_book("""
merge  1: 'e' + 's' -> 'es'  (seen 9 times)
merge  2: 'es' + 't' -> 'est'  (seen 9 times)
merge  3: 'est' + '</w>' -> 'est</w>'  (seen 9 times)
merge  4: 'l' + 'o' -> 'lo'  (seen 7 times)
merge  5: 'lo' + 'w' -> 'low'  (seen 7 times)
merge  6: 'n' + 'e' -> 'ne'  (seen 6 times)
merge  7: 'ne' + 'w' -> 'new'  (seen 6 times)
merge  8: 'new' + 'est</w>' -> 'newest</w>'  (seen 6 times)
merge  9: 'low' + '</w>' -> 'low</w>'  (seen 5 times)
merge 10: 'w' + 'i' -> 'wi'  (seen 3 times)
""")

merge  1: 'e' + 's' -> 'es'  (seen 9 times)
merge  2: 'es' + 't' -> 'est'  (seen 9 times)
merge  3: 'est' + '</w>' -> 'est</w>'  (seen 9 times)
merge  4: 'l' + 'o' -> 'lo'  (seen 7 times)
merge  5: 'lo' + 'w' -> 'low'  (seen 7 times)
merge  6: 'n' + 'e' -> 'ne'  (seen 6 times)
merge  7: 'ne' + 'w' -> 'new'  (seen 6 times)
merge  8: 'new' + 'est</w>' -> 'newest</w>'  (seen 6 times)
merge  9: 'low' + '</w>' -> 'low</w>'  (seen 5 times)
merge 10: 'w' + 'i' -> 'wi'  (seen 3 times)



-> same as the book, line for line (10 lines)


To tokenize, replay the learned merges in the order they were learned. `lowest` never
appeared in training. This cell is also the notebook's stand-in for the chapter's BPE
explorer: add your own words to the list and re-run it.

In [5]:
record()
def apply_merges(word, merges):
    symbols = tuple(word) + ("</w>",)
    for pair in merges:
        symbols = tuple(merge({symbols: 1}, pair).keys())[0]
    return symbols

for w in ["lowest", "newer", "slow", "widest"]:
    print(f"{w:<8} -> {' '.join(apply_merges(w, merges))}")
compare_with_book("""
lowest   -> low est</w>
newer    -> new e r </w>
slow     -> s low</w>
widest   -> wi d est</w>
""")

lowest   -> low est</w>
newer    -> new e r </w>
slow     -> s low</w>
widest   -> wi d est</w>



-> same as the book, line for line (4 lines)


**The caveat.** The merge table was learned over *characters*, from an alphabet of only
the letters in `low`, `lower`, `newest` and `widest`. So an unseen **character** still
breaks it. The cell lists the symbols the toy tokenizer knows (its starting symbols plus
the ten merges) and checks each piece of `çay` against them.

In [6]:
pieces = apply_merges("çay", merges)
vs_book('apply_merges("çay", merges)', repr(pieces), "('ç', 'a', 'y', '</w>')", "{}")

known = {s for w in corpus for s in tuple(w) + ("</w>",)} | {a + b for a, b in merges}
print(f"\nsymbols the toy tokenizer knows ({len(known)}): {sorted(known)}")
for p in pieces:
    print(f"  {p!r:<7} {'known' if p in known else 'never seen in training'}")

apply_merges("çay", merges)            this run ('ç', 'a', 'y', '</w>')   book ('ç', 'a', 'y', '</w>')   same as the book

symbols the toy tokenizer knows (21): ['</w>', 'd', 'e', 'es', 'est', 'est</w>', 'i', 'l', 'lo', 'low', 'low</w>', 'n', 'ne', 'new', 'newest</w>', 'o', 'r', 's', 't', 'w', 'wi']
  'ç'     never seen in training
  'a'     never seen in training
  'y'     never seen in training
  '</w>'  known


**Reading it.** `ç`, `a` and `y` never occurred in the training words, so no merge can
touch them. The fourth piece is the end-of-word marker `</w>`, which the procedure appends
to every word. Byte-level tokenizers close this hole by starting from the 256 possible
**bytes** of UTF-8 rather than from characters; section 2 shows how.

## 2. Characters or bytes: what BPE starts from

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/08-pretraining-landscape.html#sec-text-as-data-bytes). Four terms have to be told apart first.
A *character* is a letter or sign as you see it on screen. A computer stores only numbers,
so every character needs one. *ASCII* (1963) has 128 of them, for the English letters,
digits and punctuation. *Unicode* extends the idea to every writing system; the number is
the character's *code point*. A *byte* holds a number from 0 to 255, and the *UTF-8*
encoding writes each code point as one to four bytes.

The chapter puts all of that in one small table for the Turkish word `çay` (tea). The cell
rebuilds the table and checks its numbers.

**What to look at:** three characters, three code points, four bytes.

In [7]:
shown = "çay"
print(f"{'':<22}" + "".join(f"{ch:>15}" for ch in shown))
print(f"{'Unicode code point':<22}" + "".join(f"{f'{ord(ch)} (U+{ord(ch):04X})':>15}" for ch in shown))
print(f"{'in the ASCII table?':<22}" + "".join(f"{'yes' if ord(ch) < 128 else 'no':>15}" for ch in shown))
print(f"{'UTF-8 bytes':<22}" + "".join(f"{', '.join(str(b) for b in ch.encode('utf-8')):>15}" for ch in shown))
print()
vs_book('characters: len("çay")', len(shown), 3)
vs_book('bytes: len("çay".encode("utf-8"))', len(shown.encode("utf-8")), 4)
vs_book("code point of ç", ord("ç"), 231)
vs_book("UTF-8 bytes of ç", list("ç".encode("utf-8")), [195, 167], "{}")

                                    ç              a              y
Unicode code point       231 (U+00E7)    97 (U+0061)   121 (U+0079)
in the ASCII table?                no            yes            yes
UTF-8 bytes                  195, 167             97            121

characters: len("çay")                 this run         3   book         3   same as the book
bytes: len("çay".encode("utf-8"))      this run         4   book         4   same as the book
code point of ç                        this run       231   book       231   same as the book
UTF-8 bytes of ç                       this run [195, 167]   book [195, 167]   same as the book


UTF-8 needs one byte for an ASCII letter, two for the Turkish letters outside ASCII, three
for most Chinese characters and four for most emoji. The code is the chapter's.

In [8]:
record()
for ch in ["a", "ç", "ş", "İ", "世", "😀"]:
    b = ch.encode("utf-8")
    print(f"{ch}  U+{ord(ch):04X}  {len(b)} byte(s): {' '.join(f'{x:02X}' for x in b)}")
compare_with_book("""
a  U+0061  1 byte(s): 61
ç  U+00E7  2 byte(s): C3 A7
ş  U+015F  2 byte(s): C5 9F
İ  U+0130  2 byte(s): C4 B0
世  U+4E16  3 byte(s): E4 B8 96
😀  U+1F600  4 byte(s): F0 9F 98 80
""")

a  U+0061  1 byte(s): 61
ç  U+00E7  2 byte(s): C3 A7
ş  U+015F  2 byte(s): C5 9F
İ  U+0130  2 byte(s): C4 B0
世  U+4E16  3 byte(s): E4 B8 96
😀  U+1F600  4 byte(s): F0 9F 98 80



-> same as the book, line for line (6 lines)


**Starting from bytes.** A tokenizer that starts from bytes has a complete alphabet from the
first step, because every text in every language is a sequence of the 256 byte values. Here
is OpenAI's byte-level `cl100k_base`, showing the bytes behind each token. The code is the
chapter's.

**What to look at:** `ç` is one token, `ŞİŞE` costs twice what `şişe` does, and nothing
becomes `<UNK>`.

In [9]:
record()
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
for text in ["çay", "şişe", "ŞİŞE", "😀"]:
    pieces = [enc.decode_single_token_bytes(t) for t in enc.encode(text)]
    print(f"{text:<5} {len(pieces)} tokens  {pieces}")
compare_with_book("""
çay   2 tokens  [b'\\xc3\\xa7', b'ay']
şişe  3 tokens  [b'\\xc5\\x9fi', b'\\xc5\\x9f', b'e']
ŞİŞE  6 tokens  [b'\\xc5', b'\\x9e', b'\\xc4\\xb0', b'\\xc5', b'\\x9e', b'E']
😀     2 tokens  [b'\\xf0\\x9f\\x98', b'\\x80']
""")

çay   2 tokens  [b'\xc3\xa7', b'ay']
şişe  3 tokens  [b'\xc5\x9fi', b'\xc5\x9f', b'e']
ŞİŞE  6 tokens  [b'\xc5', b'\x9e', b'\xc4\xb0', b'\xc5', b'\x9e', b'E']
😀     2 tokens  [b'\xf0\x9f\x98', b'\x80']



-> same as the book, line for line (4 lines)


**But bytes do not respect characters.** The chapter's correction: byte-level BPE merges
frequent *byte* sequences, and the merges are not bound to Unicode character boundaries.
Most frequent characters do end up as one token, but a token can also be several
characters, or only part of one. The cell below decodes each token's bytes on its own and
reports whether those bytes are valid UTF-8.

In [10]:
import tiktoken

def boundary_report(enc_name, strings):
    enc = tiktoken.get_encoding(enc_name)
    print(f"{enc_name}")
    for s in strings:
        parts = [enc.decode_single_token_bytes(t) for t in enc.encode(s)]
        kind = []
        for b in parts:
            try:
                b.decode("utf-8")
                kind.append("whole")
            except UnicodeDecodeError:
                kind.append("PARTIAL")
        print(f"  {s:<22} {len(parts):>2} tokens  {kind}")
        print(f"  {'':<22}    {parts}")
    print()


for name in ["cl100k_base", "o200k_base"]:
    boundary_report(name, ["çay", "😀", "🇹🇷", "世界"])

cl100k_base
  çay                     2 tokens  ['whole', 'whole']
                            [b'\xc3\xa7', b'ay']
  😀                       2 tokens  ['PARTIAL', 'PARTIAL']
                            [b'\xf0\x9f\x98', b'\x80']
  🇹🇷                      6 tokens  ['PARTIAL', 'PARTIAL', 'PARTIAL', 'PARTIAL', 'PARTIAL', 'PARTIAL']
                            [b'\xf0\x9f', b'\x87', b'\xb9', b'\xf0\x9f', b'\x87', b'\xb7']
  世界                      3 tokens  ['PARTIAL', 'PARTIAL', 'whole']
                            [b'\xe4\xb8', b'\x96', b'\xe7\x95\x8c']

o200k_base
  çay                     2 tokens  ['whole', 'whole']
                            [b'\xc3\xa7', b'ay']
  😀                       1 tokens  ['whole']
                            [b'\xf0\x9f\x98\x80']
  🇹🇷                      4 tokens  ['PARTIAL', 'PARTIAL', 'PARTIAL', 'PARTIAL']
                            [b'\xf0\x9f\x87', b'\xb9', b'\xf0\x9f\x87', b'\xb7']
  世界                      1 tokens  ['whole']
                    

**Reading it.** `çay` splits into two tokens and both are valid UTF-8 — the Turkish `ç`
did become a single token, which is the usual case for a frequent character. The emoji is
the counter-example: under `cl100k_base`, `😀` is two tokens and *neither* is a valid
character on its own; the merge table cut through the middle of a four-byte sequence. The
Turkish flag `🇹🇷` is six such pieces. `o200k_base`, trained later and with a larger
vocabulary, keeps `😀` whole but still splits the flag.

This is why "frequent byte pairs merge back into whole characters first" is the wrong
mental model. Nothing in the algorithm knows where a character ends. It matters in
practice when you slice a token sequence: cutting a prompt at an arbitrary token can leave
you holding half a character.

**Byte fallback.** A third design combines the two. The vocabulary is built from characters,
but one token for each of the 256 byte values is added to it. When the tokenizer meets a
character it does not know, it writes that character's UTF-8 bytes with those tokens
instead of writing `<UNK>`. The chapter trains SentencePiece twice on eight Turkish words
with no `ç` in them, once without the option and once with it, and gives both models the
word `çay`.

The code is the chapter's with one change: the chapter writes its three small files to the
working directory, and this cell writes them under `data/cache/` so they stay out of the
repository.

**What to look at:** the end of each line, which is the tokens turned back into text.

In [11]:
record()
import sentencepiece as spm

mini = CACHE / "sentencepiece-mini"                 # not in the book: keep the files in the cache
mini.mkdir(parents=True, exist_ok=True)

words = ["ev", "evde", "evden", "evler", "yol", "yolda", "yoldan", "yollar"]
with open(mini / "mini.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(words * 5) + "\n")            # no 'ç' anywhere in this text

for fallback in [False, True]:
    spm.SentencePieceTrainer.train(
        input=str(mini / "mini.txt"), model_prefix=str(mini / "mini"), model_type="bpe",
        vocab_size=20 + (256 if fallback else 0), byte_fallback=fallback,
        character_coverage=1.0, hard_vocab_limit=False, minloglevel=2,
        bos_id=-1, eos_id=-1, pad_id=-1)
    sp = spm.SentencePieceProcessor(model_file=str(mini / "mini.model"))
    ids = sp.encode("çay")
    print(f"byte_fallback={fallback!s:<5}  vocabulary {sp.get_piece_size():>3}  "
          f"{sp.encode('çay', out_type=str)}  ->  {sp.decode(ids)!r}")
compare_with_book("""
byte_fallback=False  vocabulary  20  ['▁', 'ç', 'a', 'y']  ->  ' ⁇ ay'
byte_fallback=True   vocabulary 276  ['▁', '<0xC3>', '<0xA7>', 'a', 'y']  ->  'çay'
""")

byte_fallback=False  vocabulary  20  ['▁', 'ç', 'a', 'y']  ->  ' ⁇ ay'
byte_fallback=True   vocabulary 276  ['▁', '<0xC3>', '<0xA7>', 'a', 'y']  ->  'çay'



-> same as the book, line for line (2 lines)


**Reading it.** Without the option, `ç` gets the "unknown" id and comes back as `⁇`: the
letter is gone. With it, the vocabulary has grown by the 256 byte tokens (from 20 to 276),
and `ç` is written as `<0xC3>` `<0xA7>`. Those are the two bytes from the `çay` table
above: hexadecimal C3 is 195 and A7 is 167. Turned back into text, `çay` is intact. The
leading `▁` marks where a word starts; section 3 comes back to it.

## 3. Who pays for Turkish

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/08-pretraining-landscape.html#sec-text-as-data-turkish). Article 1 of the Universal Declaration
of Human Rights, in its official English and Turkish texts (published by the United
Nations), run through two of OpenAI's tokenizers. The code is the chapter's. Lengths are
counted in Unicode code points, which is what Python's `len` returns for a string.

The first call downloads each encoding's vocabulary file from OpenAI's public blob storage
and caches it under `data/cache/tiktoken`. The chapter says every measurement in this
section was taken after Unicode NFC normalization, so the cell checks that both strings are
already in NFC.

In [12]:
record()
import tiktoken

en = ("All human beings are born free and equal in dignity and rights. "
      "They are endowed with reason and conscience and should act towards "
      "one another in a spirit of brotherhood.")
tr = ("Bütün insanlar hür, haysiyet ve haklar bakımından eşit doğarlar. "
      "Akıl ve vicdana sahiptirler ve birbirlerine karşı kardeşlik "
      "zihniyeti ile hareket etmelidirler.")

for name in ["cl100k_base", "o200k_base"]:
    enc = tiktoken.get_encoding(name)
    a, b = len(enc.encode(en)), len(enc.encode(tr))
    print(f"{name}:")
    print(f"  english : {len(en):>4} code points -> {a:>3} tokens  "
          f"({len(en) / a:4.2f} code points/token)")
    print(f"  turkish : {len(tr):>4} code points -> {b:>3} tokens  "
          f"({len(tr) / b:4.2f} code points/token)")
    print(f"  Turkish/English token count ratio: {b / a:4.2f}")
compare_with_book("""
cl100k_base:
  english :  170 code points ->  33 tokens  (5.15 code points/token)
  turkish :  160 code points ->  60 tokens  (2.67 code points/token)
  Turkish/English token count ratio: 1.82
o200k_base:
  english :  170 code points ->  33 tokens  (5.15 code points/token)
  turkish :  160 code points ->  46 tokens  (3.48 code points/token)
  Turkish/English token count ratio: 1.39
""")
print("both strings already NFC:", unicodedata.is_normalized("NFC", en) and unicodedata.is_normalized("NFC", tr))

cl100k_base:
  english :  170 code points ->  33 tokens  (5.15 code points/token)
  turkish :  160 code points ->  60 tokens  (2.67 code points/token)
  Turkish/English token count ratio: 1.82
o200k_base:
  english :  170 code points ->  33 tokens  (5.15 code points/token)
  turkish :  160 code points ->  46 tokens  (3.48 code points/token)
  Turkish/English token count ratio: 1.39



-> same as the book, line for line (8 lines)
both strings already NFC: True


**One Turkish word, four suffixes.** The chapter prints how the tokenizer splits `ev`
("house") as suffixes stack up, but not the code. The leading space in `' ev'` says the
word was encoded after a space, as it would appear mid-sentence. **Our reading:**
`cl100k_base`, each word preceded by a space. The cell also counts the English translation,
*from our houses*, which the chapter says costs three tokens, and checks the chapter's remark
that the newer `o200k_base` keeps `imiz` in one piece.

In [13]:
enc = tiktoken.get_encoding("cl100k_base")

record()
for w in ["ev", "evler", "evlerimiz", "evlerimizden"]:
    ids = enc.encode(" " + w)
    print(f"{w:<14} {len(ids)} tokens  {[enc.decode([i]) for i in ids]}")
compare_with_book("""
ev             1 tokens  [' ev']
evler          2 tokens  [' ev', 'ler']
evlerimiz      4 tokens  [' ev', 'ler', 'im', 'iz']
evlerimizden   5 tokens  [' ev', 'ler', 'im', 'iz', 'den']
""")
print()
vs_book('tokens in " from our houses"', len(enc.encode(" from our houses")), 3)
o200k = tiktoken.get_encoding("o200k_base")
vs_book("o200k_base splits evlerimizden as",
        "|".join(o200k.decode([i]) for i in o200k.encode(" evlerimizden")).strip(), "ev|ler|imiz|den", "{}")

ev             1 tokens  [' ev']
evler          2 tokens  [' ev', 'ler']
evlerimiz      4 tokens  [' ev', 'ler', 'im', 'iz']
evlerimizden   5 tokens  [' ev', 'ler', 'im', 'iz', 'den']



-> same as the book, line for line (4 lines)

tokens in " from our houses"           this run         3   book         3   same as the book
o200k_base splits evlerimizden as      this run ev|ler|imiz|den   book ev|ler|imiz|den   same as the book


**Reading it.** `ev|ler|im|iz|den` lines up with the real morphology at three of its four
boundaries. The asterisk the chapter puts on it is `im|iz`: *-imiz* ("our") is a single
morpheme that BPE cut in half, because `im` and `iz` are each frequent on their own.
`o200k_base` does not make that cut, which shows that where the cut falls depends on the
merge table, not on the language.

**Where did the spaces go?** The first token above is `' ev'`, with a space in front. GPT's
BPE tokenizers and BERT's WordPiece first cut the text into words at whitespace and
punctuation (*pre-tokenization*) and only then split each word into subwords, so no token
spans two words. To turn tokens back into text, where a word starts has to be written on
the token itself, and three tokenizer families do that with three different marks. The
chapter's table shows the same two words, `evlerimizden geldik` ("we came from our
houses"), in all three. It gives no code. **Our reading:** the three tokenizers the table
names, with `␣` printed for the space as the table does.

The cell downloads two tokenizer files from the Hugging Face Hub and caches them under
`data/cache/huggingface`: multilingual BERT's (2 MB, Apache-2.0) and XLM-R's (9 MB, MIT).

In [14]:
from tokenizers import Tokenizer

two_words = "evlerimizden geldik"
gpt = tiktoken.get_encoding("cl100k_base")
xlmr = Tokenizer.from_pretrained("FacebookAI/xlm-roberta-base")
mbert = Tokenizer.from_pretrained("google-bert/bert-base-multilingual-cased")

rows = [
    ("GPT (cl100k_base)",
     [gpt.decode([t]).replace(" ", "␣") for t in gpt.encode(two_words)],
     "ev ler im iz den ␣geld ik"),
    ("SentencePiece (XLM-R)",
     xlmr.encode(two_words, add_special_tokens=False).tokens,
     "▁ev lerimiz den ▁geldi k"),
    ("WordPiece (multilingual BERT)",
     mbert.encode(two_words, add_special_tokens=False).tokens,
     "ev ##leri ##mi ##zde ##n geldi ##k"),
]
for family, tokens, book in rows:
    vs_book(family, " ".join(tokens), book, "{}")

GPT (cl100k_base)                      this run ev ler im iz den ␣geld ik   book ev ler im iz den ␣geld ik   same as the book
SentencePiece (XLM-R)                  this run ▁ev lerimiz den ▁geldi k   book ▁ev lerimiz den ▁geldi k   same as the book
WordPiece (multilingual BERT)          this run ev ##leri ##mi ##zde ##n geldi ##k   book ev ##leri ##mi ##zde ##n geldi ##k   same as the book


**Reading it.** GPT glues the space onto the first token of the *next* word (`␣geld`).
SentencePiece puts a visible `▁` at the start of a word. WordPiece marks the opposite case:
it puts `##` in front of every piece that *continues* a word. All three carry the same
information: a new word starts here.

**The four stages of a tokenizer.** Text goes through four stages before it reaches the
model: normalization, pre-tokenization, subword encoding and special tokens. The chapter
follows one Turkish sentence ("We drank tea, then we left our houses.") through the
tokenizer of `bert-base-multilingual-uncased` (a 2 MB file, Apache-2.0). The code is the
chapter's.

**What to look at:** line 1, where `Çay` becomes `cay`.

In [15]:
record()
from tokenizers import Tokenizer

tok = Tokenizer.from_pretrained("google-bert/bert-base-multilingual-uncased")
text = "Çay içtik, sonra evlerimizden çıktık."

norm = tok.normalizer.normalize_str(text)
print("1 normalize    :", norm)
print("2 pre-tokenize :", [w for w, _ in tok.pre_tokenizer.pre_tokenize_str(norm)])
print("3 subwords     :", tok.encode(text, add_special_tokens=False).tokens)
print("4 special      :", tok.encode(text).tokens)
print("  ids          :", tok.encode(text).ids)
compare_with_book("""
1 normalize    : cay ictik, sonra evlerimizden cıktık.
2 pre-tokenize : ['cay', 'ictik', ',', 'sonra', 'evlerimizden', 'cıktık', '.']
3 subwords     : ['cay', 'ic', '##tik', ',', 'sonra', 'ev', '##leri', '##mi', '##zde', '##n', 'cıktı', '##k', '.']
4 special      : ['[CLS]', 'cay', 'ic', '##tik', ',', 'sonra', 'ev', '##leri', '##mi', '##zde', '##n', 'cıktı', '##k', '.', '[SEP]']
  ids          : [101, 21373, 11797, 13537, 117, 12624, 18110, 12737, 10555, 55567, 10115, 59161, 10167, 119, 102]
""")
print()
vs_book("the cased model leaves the text as it is", mbert.normalizer.normalize_str(text) == text, True, "{}")

1 normalize    : cay ictik, sonra evlerimizden cıktık.
2 pre-tokenize : ['cay', 'ictik', ',', 'sonra', 'evlerimizden', 'cıktık', '.']
3 subwords     : ['cay', 'ic', '##tik', ',', 'sonra', 'ev', '##leri', '##mi', '##zde', '##n', 'cıktı', '##k', '.']
4 special      : ['[CLS]', 'cay', 'ic', '##tik', ',', 'sonra', 'ev', '##leri', '##mi', '##zde', '##n', 'cıktı', '##k', '.', '[SEP]']
  ids          : [101, 21373, 11797, 13537, 117, 12624, 18110, 12737, 10555, 55567, 10115, 59161, 10167, 119, 102]



-> same as the book, line for line (5 lines)

the cased model leaves the text as it is this run      True   book      True   same as the book


**Reading it.** Each line is one stage. Normalization lowercased the text and stripped the
marks from the letters, so the model can no longer tell `çay` (tea) from `cay`; `ı`
survived because Unicode defines it as a letter of its own. Pre-tokenization made the comma
and the full stop pieces of their own. Subword encoding ran inside each word:
`evlerimizden` is five pieces and `sonra` stays whole. The last stage added `[CLS]` and
`[SEP]`, and the integers are what the model receives. The case-sensitive version of the
same model leaves the text untouched in the first stage, as the chapter says.

**One practical detail: NFC.** A letter with a mark, such as `ö`, can be stored as a single
code point or as `o` followed by a separate combining mark. The two look the same on
screen, but a tokenizer counts them differently. So the chapter's advice is one line before
any tokenizer: `unicodedata.normalize("NFC", text)`. The chapter gives no numbers for this;
the ones below are this run's, on one word.

In [16]:
shown = "görüşebileceklerimiz"
nfc, nfd = unicodedata.normalize("NFC", shown), unicodedata.normalize("NFD", shown)
print(f"NFC: {nfc}   {len(nfc)} code points, {len(nfc.encode('utf-8'))} bytes")
print(f"NFD: {nfd}   {len(nfd)} code points, {len(nfd.encode('utf-8'))} bytes")
print(f"equal in Python: {nfc == nfd}")
for name in ["cl100k_base", "o200k_base"]:
    e = tiktoken.get_encoding(name)
    print(f"{name:<12} NFC {len(e.encode(nfc)):>2} tokens   NFD {len(e.encode(nfd)):>2} tokens")
print(f"equal after normalize('NFC', ...): {unicodedata.normalize('NFC', nfd) == nfc}")

NFC: görüşebileceklerimiz   20 code points, 23 bytes
NFD: görüşebileceklerimiz   23 code points, 26 bytes
equal in Python: False
cl100k_base  NFC 10 tokens   NFD 15 tokens
o200k_base   NFC  7 tokens   NFD 10 tokens
equal after normalize('NFC', ...): True


**Reading it.** The two spellings look the same, compare as unequal, and the decomposed one
costs more tokens under both encodings. One call makes them the same string again.

**Why not just use characters?** The chapter's intuition check counts the same Turkish
sentence in three unit sizes and squares each count, because self-attention compares every
unit with every other. The code is the chapter's; `tr` is the Turkish text of Article 1 from
the cell above.

**What to look at:** the middle column, and then the column next to it.

In [17]:
record()
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
units = {
    "code points": list(tr),
    "subword tokens": [enc.decode([t]) for t in enc.encode(tr)],
    "words": tr.split(),
}
for name, seq in units.items():
    n = len(seq)
    print(f"{name:<15} {n:>3} units  {n * n:>6,} pairs  first six: {seq[:6]}")
compare_with_book("""
code points     160 units  25,600 pairs  first six: ['B', 'ü', 't', 'ü', 'n', ' ']
subword tokens   60 units   3,600 pairs  first six: ['B', 'üt', 'ün', ' insan', 'lar', ' h']
words            21 units     441 pairs  first six: ['Bütün', 'insanlar', 'hür,', 'haysiyet', 've', 'haklar']
""")

code points     160 units  25,600 pairs  first six: ['B', 'ü', 't', 'ü', 'n', ' ']
subword tokens   60 units   3,600 pairs  first six: ['B', 'üt', 'ün', ' insan', 'lar', ' h']
words            21 units     441 pairs  first six: ['Bütün', 'insanlar', 'hür,', 'haysiyet', 've', 'haklar']



-> same as the book, line for line (3 lines)


**Reading it.** Code points give the longest sequence, 160 units and 25,600 pairs; whole
words give the shortest, 21 units, but a word-level vocabulary meets forms it has never seen.
Subwords sit between the two. The numbers belong to this sentence and this tokenizer.

## 4. Characters, subwords, whole words: one example

[Chapter section](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/08-pretraining-landscape.html#sec-text-as-data-granularities). One small corpus, three tokenizers.
The corpus is built from twelve Turkish stems, each in five forms: 60 forms and 216
occurrences. The two test words, `evlerden` ("from the houses") and `okullardan` ("from the
schools"), are not among them. The code is the chapter's; it reuses `pair_counts`, `merge`
and `apply_merges` from section 1.

**What to look at:** the number of units each tokenizer needs for the two unseen words.

In [18]:
record()
stems = {"ev": "e", "el": "e", "göl": "e", "köy": "e", "gün": "e", "yüz": "e",
         "okul": "a", "yol": "a", "kol": "a", "yıl": "a", "kız": "a", "masa": "a"}
corpus = {}
for stem, v in stems.items():          # v: the vowel the suffixes take
    for suffix, n in [("", 4), (f"l{v}r", 2), (f"d{v}", 4),
                      (f"d{v}n", 4), (f"l{v}rd{v}", 4)]:
        corpus[stem + suffix] = n      # ev, evler, evde, evden, evlerde, ...
test = ["evlerden", "okullardan"]      # neither word is in the corpus

chars = sorted({c for word in corpus for c in word})
words = {tuple(w) + ("</w>",): n for w, n in corpus.items()}
merges = []
while len(chars) + 1 + len(merges) < 60:   # stop at a 60-symbol vocabulary
    best, _ = pair_counts(words).most_common(1)[0]
    merges.append(best)
    words = merge(words, best)

tokenizers = {
    "whole words": (len(corpus), lambda w: [w if w in corpus else "<UNK>"]),
    "characters": (len(chars), lambda w: list(w)),
    "subwords (BPE)": (len(chars) + 1 + len(merges),
                       lambda w: list(apply_merges(w, merges))),
}
for name, (size, split) in tokenizers.items():
    pieces = [p for w in test for p in split(w)]
    print(f"{name:<15} vocabulary {size:>2}   {len(pieces):>2} units: {' '.join(pieces)}")
compare_with_book("""
whole words     vocabulary 60    2 units: <UNK> <UNK>
characters      vocabulary 18   18 units: e v l e r d e n o k u l l a r d a n
subwords (BPE)  vocabulary 60    6 units: ev ler den</w> okul lar dan</w>
""")
print()
vs_book("forms in the corpus", len(corpus), 60)
vs_book("occurrences in the corpus", sum(corpus.values()), 216)

whole words     vocabulary 60    2 units: <UNK> <UNK>
characters      vocabulary 18   18 units: e v l e r d e n o k u l l a r d a n
subwords (BPE)  vocabulary 60    6 units: ev ler den</w> okul lar dan</w>



-> same as the book, line for line (3 lines)

forms in the corpus                    this run        60   book        60   same as the book
occurrences in the corpus              this run       216   book       216   same as the book


**Reading it.** The whole-word tokenizer has 60 entries and still answers `<UNK>` twice. The
character tokenizer needs only 18 symbols and loses nothing, but spends 18 units on two
words. BPE, stopped at the same 60 entries as the word list, covers both words with six
pieces, and the pieces are ones it learned from other words.

### Deep dive: WordPiece and Unigram LM on the same corpus

The chapter's collapsed deep dive trains two more subword learners on this corpus, in a few
dozen lines each. They are teaching implementations: WordPiece here merges the pair with the
highest score $\frac{\text{count}(ab)}{\text{count}(a)\,\text{count}(b)}$ instead of the
highest count, and Unigram LM starts from every substring and prunes. The code is the
chapter's.

In [19]:
import math

def train_wordpiece(corpus, size):
    words = {tuple([w[0]] + ["##" + c for c in w[1:]]): n for w, n in corpus.items()}
    vocab = {s for symbols in words for s in symbols}
    first = None
    while len(vocab) < size:
        pairs, single = Counter(), Counter()
        for symbols, n in words.items():
            for s in symbols:
                single[s] += n
            for a, b in zip(symbols, symbols[1:]):
                pairs[(a, b)] += n
        score = {p: c / (single[p[0]] * single[p[1]]) for p, c in pairs.items()}
        best = max(score, key=lambda p: (score[p], p))
        first = first or (pairs, score)
        glued = best[0] + best[1].removeprefix("##")
        new = {}
        for symbols, n in words.items():
            out, i = [], 0
            while i < len(symbols):
                if symbols[i:i + 2] == best:
                    out.append(glued)
                    i += 2
                else:
                    out.append(symbols[i])
                    i += 1
            new[tuple(out)] = n
        words, vocab = new, vocab | {glued}
    return vocab, first

def wordpiece_split(word, vocab):          # greedy: longest known piece first
    pieces, start = [], 0
    while start < len(word):
        for end in range(len(word), start, -1):
            piece = word[start:end] if start == 0 else "##" + word[start:end]
            if piece in vocab:
                break
        else:
            return ["[UNK]"]
        pieces.append(piece)
        start = end
    return pieces

def best_split(word, logp):                # Viterbi: the most probable split
    best = [(0.0, [])] + [(-math.inf, None)] * len(word)
    for end in range(1, len(word) + 1):
        for start in range(end):
            piece = word[start:end]
            if piece in logp and best[start][1] is not None:
                score = best[start][0] + logp[piece]
                if score > best[end][0]:
                    best[end] = (score, best[start][1] + [piece])
    return best[-1]

def train_unigram(corpus, size):
    counts = Counter()                     # start big: every substring
    for w, n in corpus.items():
        for i in range(len(w)):
            for j in range(i + 1, len(w) + 1):
                counts[w[i:j]] += n
    start_size = len(counts)
    logprobs = lambda c: {p: math.log(k / sum(c.values())) for p, k in c.items()}
    loss = lambda lp: -sum(n * best_split(w, lp)[0] for w, n in corpus.items())
    while len(counts) > size:
        lp = logprobs(counts)
        base = loss(lp)
        cost = {p: loss({q: v for q, v in lp.items() if q != p}) - base
                for p in counts if len(p) > 1}     # single letters always stay
        k = max(1, min(len(counts) - size, len(cost) // 5))
        for p in sorted(cost, key=lambda p: (cost[p], p))[:k]:
            del counts[p]                  # prune the pieces that cost least
        lp = logprobs(counts)              # re-estimate from the best splits
        counts = Counter({p: 0.001 for p in counts})
        for w, n in corpus.items():
            for piece in best_split(w, lp)[1]:
                counts[piece] += n
    return logprobs(counts), start_size

In [20]:
record()
wp_vocab, (pairs, score) = train_wordpiece(corpus, 60)
uni_logp, start_size = train_unigram(corpus, 60)

print("first step, WordPiece initial pairs:")
print("  highest raw pair counts:  ",
      [(a + b.removeprefix("##"), c) for (a, b), c in pairs.most_common(3)])
print("  highest score (WordPiece):",
      [(a + b.removeprefix("##"), round(s, 3))
       for (a, b), s in sorted(score.items(), key=lambda x: -x[1])[:3]])
print(f"  Unigram LM starts from {start_size} substrings and prunes to {len(uni_logp)}")
print()
for w in test:
    print(f"{w:<11} BPE       : {' | '.join(apply_merges(w, merges))}")
    print(f"{'':<11} WordPiece : {' | '.join(wordpiece_split(w, wp_vocab))}")
    print(f"{'':<11} Unigram LM: {' | '.join(best_split(w, uni_logp)[1])}")
compare_with_book("""
first step, WordPiece initial pairs:
  highest raw pair counts:   [('##de', 72), ('##da', 72), ('##rd', 48)]
  highest score (WordPiece): [('ok', 0.056), ('##ku', 0.056), ('ev', 0.028)]
  Unigram LM starts from 322 substrings and prunes to 60

evlerden    BPE       : ev | ler | den</w>
            WordPiece : ev | ##l | ##e | ##r | ##d | ##e | ##n
            Unigram LM: evlerde | n
okullardan  BPE       : okul | lar | dan</w>
            WordPiece : okullar | ##d | ##a | ##n
            Unigram LM: okullarda | n
""")

first step, WordPiece initial pairs:
  highest raw pair counts:   [('##de', 72), ('##da', 72), ('##rd', 48)]
  highest score (WordPiece): [('ok', 0.056), ('##ku', 0.056), ('ev', 0.028)]
  Unigram LM starts from 322 substrings and prunes to 60

evlerden    BPE       : ev | ler | den</w>
            WordPiece : ev | ##l | ##e | ##r | ##d | ##e | ##n
            Unigram LM: evlerde | n
okullardan  BPE       : okul | lar | dan</w>
            WordPiece : okullar | ##d | ##a | ##n
            Unigram LM: okullarda | n



-> same as the book, line for line (11 lines)


**Reading it.** The first three lines show the difference in the criterion: the pairs that
occur most often are not the pairs with the highest WordPiece score. The three learners then
split the two test words differently, from the same corpus and the same vocabulary size.

Unigram LM does not store merges; it stores a probability for every piece and picks the
split with the highest product. The next cell lists every split of `evlerden` this
vocabulary allows, with its score.

In [21]:
record()
def every_split(word, logp, prefix=()):
    if not word:
        yield sum(logp[p] for p in prefix), list(prefix)
    for end in range(1, len(word) + 1):
        if word[:end] in logp:
            yield from every_split(word[end:], logp, prefix + (word[:end],))

splits = sorted(every_split("evlerden", uni_logp), reverse=True)
print(f"{len(splits)} ways to split 'evlerden' with this vocabulary:")
for logp, pieces in splits:
    print(f"  score = {math.exp(logp):.1e}   {' | '.join(pieces)}")
compare_with_book("""
4 ways to split 'evlerden' with this vocabulary:
  score = 2.1e-03   evlerde | n
  score = 1.2e-14   evler | d | e | n
  score = 1.3e-22   ev | l | e | r | d | e | n
  score = 1.1e-31   e | v | l | e | r | d | e | n
""")

4 ways to split 'evlerden' with this vocabulary:
  score = 2.1e-03   evlerde | n
  score = 1.2e-14   evler | d | e | n
  score = 1.3e-22   ev | l | e | r | d | e | n
  score = 1.1e-31   e | v | l | e | r | d | e | n



-> same as the book, line for line (5 lines)


**Checking against SentencePiece.** The chapter trains the real Unigram implementation on the
same corpus and gives it the same two words. The code is the chapter's with one change: its
files go under `data/cache/` instead of the working directory. The cell then checks the
facts the chapter states about this run.

In [22]:
record()
import sentencepiece as spm

lines = [w for w, n in corpus.items() for _ in range(n)]
mini = CACHE / "sentencepiece-mini"                 # not in the book: keep the files in the cache
mini.mkdir(parents=True, exist_ok=True)
with open(mini / "corpus.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")

spm.SentencePieceTrainer.train(
    input=str(mini / "corpus.txt"), model_prefix=str(mini / "uni"), model_type="unigram", vocab_size=60,
    character_coverage=1.0, hard_vocab_limit=False, minloglevel=2,
    bos_id=-1, eos_id=-1, pad_id=-1)
uni = spm.SentencePieceProcessor(model_file=str(mini / "uni.model"))
for w in test:
    print(f"{w:<11} {' | '.join(uni.encode(w, out_type=str))}")
compare_with_book("""
evlerden    ▁evlerde | n
okullardan  ▁okul | l | a | r | d | a | n
""")
print()
pieces = [uni.id_to_piece(i) for i in range(uni.get_piece_size())]
vs_book("entries, including <unk>", len(pieces), 60)
vs_book("<unk> is one of them", "<unk>" in pieces, True, "{}")
vs_book("vocabulary contains ▁okul", "▁okul" in pieces, True, "{}")
vs_book("vocabulary contains ▁evlerde", "▁evlerde" in pieces, True, "{}")
vs_book("vocabulary contains ▁okullarda", "▁okullarda" in pieces, False, "{}")

evlerden    ▁evlerde | n
okullardan  ▁okul | l | a | r | d | a | n



-> same as the book, line for line (2 lines)

entries, including <unk>               this run        60   book        60   same as the book
<unk> is one of them                   this run      True   book      True   same as the book
vocabulary contains ▁okul              this run      True   book      True   same as the book
vocabulary contains ▁evlerde           this run      True   book      True   same as the book
vocabulary contains ▁okullarda         this run     False   book     False   same as the book


**Reading it.** SentencePiece and the teaching version agree on `evlerden` and differ in the
details of the other word; their training procedures are not the same. The chapter's caution
applies to all of this section: these are sixty-entry vocabularies on a corpus of 216 word
occurrences, built to show the mechanics, not to compare the algorithms.

## 5. When WordPiece says `[UNK]`

[Interview questions](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/08-pretraining-landscape.html#sec-pretraining-interview). The chapter's interview answer on BPE notes that a
byte-level alphabet is not the only thing that decides whether a tokenizer emits an unknown
token: in multilingual BERT's tokenizer a word longer than the implementation's limit becomes
`[UNK]` even when every character is known. The chapter gives no code. **Our reading:** the
same `bert-base-multilingual-cased` tokenizer file as in section 3, on a word of 100 and of
101 letters `a`.

In [23]:
from tokenizers import Tokenizer

mbert = Tokenizer.from_pretrained("bert-base-multilingual-cased")
for n, book_pieces, book_first in [(100, 50, "aa"), (101, 1, "[UNK]")]:
    pieces = mbert.encode("a" * n, add_special_tokens=False).tokens
    vs_book(f"pieces for a word of {n} letters", len(pieces), book_pieces)
    vs_book("  the first piece", pieces[0], book_first, "{}")

pieces for a word of 100 letters       this run        50   book        50   same as the book
  the first piece                      this run        aa   book        aa   same as the book
pieces for a word of 101 letters       this run         1   book         1   same as the book
  the first piece                      this run     [UNK]   book     [UNK]   same as the book


## 6. Exercises

These follow [chapter 8's own exercises](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/08-pretraining-landscape.html#exercises) 11 and 12, starting from the cells above.

1. **Tokenizer economics.** Replace the two strings in section 3 with a paragraph of your
   own technical writing in Turkish and its English translation, and count tokens under
   `cl100k_base` and `o200k_base`. First compute the ratio of token counts for the same
   content. Then estimate the annual cost at 100,000 requests per month with the unit prices
   of a model you choose, stating their date; report input and output tokens separately, and
   state the assumption that one paragraph stands for all requests. Before you trust the
   ratio, count a few of your own words with and without a space in front, as the `ev` cell
   does: does the space change your answer?

2. **Measure the Turkish token cost over many examples.** Section 3 used one short pair of
   texts. Collect 50 to 100 sentence pairs from a parallel text you have and, for each
   tokenizer, make two separate measurements: the Turkish/English ratio of token counts for
   each pair, and the tokens per 1,000 Unicode code points in each language. Report the
   median and the interquartile range, and say where the single-pair measurement falls in
   that distribution.

Word-level tokens, normalization, Zipf and Heaps, and TF-IDF are in the
[chapter 2 notebook](https://github.com/bbardakk/uygulamali-dogal-dil-isleme-notebooks/blob/main/en/chapters/02-text-as-data/02-text-as-data.ipynb).

---

*Code in this notebook is MIT-licensed; the prose is CC BY 4.0. The two Article 1 texts are
the United Nations' official English and Turkish versions of the Universal Declaration of
Human Rights. Tokenizer vocabularies are downloaded by `tiktoken` from OpenAI; the tokenizer
files of multilingual BERT, XLM-R and the other models are downloaded from the Hugging Face
Hub and keep their own licences.*